In [1]:
import requests
import torch
import re
import time
import psutil
import subprocess

import pandas as pd

from datasets import load_dataset

In [2]:
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1066 entries, 0 to 1065
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    1066 non-null   object
 1   label   1066 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 16.8+ KB


In [3]:
test['label'] = test['label'].apply(lambda x: 'positive' if x == 1 else 'negative')

labels = test['label'].unique()

test

,text,label
0,lovingly photographed in the manner of a golde...,positive
1,consistently clever and suspenseful .,positive
2,"it's like a "" big chill "" reunion of the baade...",positive
3,the story gives ample opportunity for large-sc...,positive
4,"red dragon "" never cuts corners .",positive
...,...,...
1061,a terrible movie that some people will neverth...,negative
1062,there are many definitions of 'time waster' bu...,negative
1063,"as it stands , crocodile hunter has the hurrie...",negative
1064,the thing looks like a made-for-home-video qui...,negative


In [4]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_17524\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


51220480

In [5]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [6]:
def classify(text, labels):
    url = "http://localhost:11434/api/chat"
    
    messages = [
        {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions."},
        {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis of movie reviews. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
    ]
    
    start_time = time.time()

    try:
        response = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": True,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 3100
            }
        }, timeout=30)
        response_time = time.time() - start_time
        vram_usage = get_gpu_memory_usage()
        ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)
        response = response.json()
        response_text = response['message'].get('thinking', '') if 'message' in response else ''
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return "error", {}, 0, 0, 0, 0, f"API Error: {e}"
    
    if not response.get('done', False):
        print(f"Ollama returned an incomplete response: {response.get('error')}")
        return 'error', {}, response_time, 0, 0, 0, response.get('error', 'Incomplete response')
    
    if 'message' in response and 'content' in response['message']:
        classification_text = response['message']['content'].lower()
        print("Response fields:", ', '.join(response.keys()))
        print(response)
        total_time = response['total_duration'] / 1_000_000_000
    else:
        messages.append({"role": "assistant", "content": response_text + '</think>'})
        start_time2 = time.time()
        response2 = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": False,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 430
            }
        })
        response_time += time.time() - start_time2
        response2 = response2.json()
        classification_text = response2['message']['content'].lower() if 'message' in response2 and 'content' in response2['message'] else ''
        total_time = response['total_duration'] / 1_000_000_000 + response2['total_duration'] / 1_000_000_000
    
    label_counts = {label: len(re.findall(r'\b' + re.escape(label.lower()) + r'\b', classification_text)) for label in labels}
    
    if all(count == label_counts[labels[0]] for count in label_counts.values()):
        content = 'error'
    else:
        content = max(label_counts, key=label_counts.get)
    
    print(f"Text: {text}")
    print(f"Response: {content}")
    
    return content, label_counts, response_time, vram_usage, ram_usage_bytes, total_time, response_text

In [7]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'label_counts','response_time', 'vram_usage', 'ram_usage', 'total_time', 'response_text']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_17524\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


Response fields: model, created_at, message, done_reason, done, total_duration, load_duration, prompt_eval_count, prompt_eval_duration, eval_count, eval_duration
{'model': 'deepseek-r1:1.5b', 'created_at': '2025-06-11T15:12:34.602869Z', 'message': {'role': 'assistant', 'content': 'positive', 'thinking': 'Okay, so I need to figure out whether this movie review is positive or negative based on sentiment analysis. Let me read through it again carefully.\n\nThe text says, "lovingly photographed in the manner of a golden book sprung to life, Stuart Little 2 manages sweetness largely without stickiness." Hmm, okay, so the person is talking about how the movie was photographed like a golden book, which I think are those classic books with beautiful illustrations. They mention that it\'s "sounded to life," which makes me imagine something dynamic and lively.\n\nThen they say Stuart Little 2 manages sweetness without stickiness. So, instead of using sticky materials or something that might make

In [8]:
test.to_csv('results/deepseekR1_ZS_binary2.csv', index=False)
test

,text,label,prediction,label_counts,response_time,vram_usage,ram_usage,total_time,response_text
0,lovingly photographed in the manner of a golde...,positive,positive,"{'positive': 1, 'negative': 0}",8.063008,2404,89.578125,5.954098,"Okay, so I need to figure out whether this mov..."
1,consistently clever and suspenseful .,positive,negative,"{'positive': 0, 'negative': 1}",4.024655,2393,89.792969,1.973535,"Okay, so I need to figure out whether the give..."
2,"it's like a "" big chill "" reunion of the baade...",positive,negative,"{'positive': 0, 'negative': 1}",4.201763,2391,89.824219,2.152450,"Okay, so I need to figure out whether this mov..."
3,the story gives ample opportunity for large-sc...,positive,positive,"{'positive': 1, 'negative': 0}",3.389289,2393,90.390625,1.341521,"Okay, so I need to figure out whether this mov..."
4,"red dragon "" never cuts corners .",positive,error,"{'positive': 0, 'negative': 0}",3.624889,2395,90.902344,1.568098,"Okay, so I need to figure out whether the give..."
...,...,...,...,...,...,...,...,...,...
1061,a terrible movie that some people will neverth...,negative,positive,"{'positive': 1, 'negative': 0}",4.370197,2123,79.437500,2.332630,"Okay, so I need to figure out whether this mov..."
1062,there are many definitions of 'time waster' bu...,negative,negative,"{'positive': 0, 'negative': 1}",5.799053,2117,79.437500,3.743301,"Okay, so I need to figure out whether the give..."
1063,"as it stands , crocodile hunter has the hurrie...",negative,negative,"{'positive': 0, 'negative': 1}",6.543490,2117,79.625000,4.506686,"Alright, so I need to figure out whether this ..."
1064,the thing looks like a made-for-home-video qui...,negative,positive,"{'positive': 1, 'negative': 0}",6.724670,2117,79.625000,4.672647,"Okay, so I need to figure out whether this mov..."


In [9]:
y_pred = test['prediction']
y_true = test['label']

#import acc, f1_score, precision and recall from sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.655722
F1 score: 0.706135
Precision: 0.785832
Recall: 0.655722


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [10]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 4.428632850271229
Average VRAM usage: 2324.020637898687
Average RAM usage: 79.63726840994372
Average total time: 2.381600399249531


In [11]:
# save results to txt
with open('results/deepseekR1_ZS_binary2.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')
    f.write(f'Lines classified: {len(test)}\n')